# Aula 01 - Introdução ao Python

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**04/08/2026 - Sprint 1 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula01.ipynb)

## O problema

A Louis Dreyfus Company compra milho, farelo de soja, sorgo, trigo e DDGS para a
indústria de ração. Hoje ela projeta essa demanda com coeficientes fixos. O que o
parceiro pediu foi uma cadeia de três modelos: prever a produção de proteína animal,
converter isso em demanda de ração, e desdobrar a ração nos macroingredientes.

Numa cadeia assim, erro na entrada não fica na entrada. Por isso este primeiro
laboratório não treina modelo nenhum: ele abre o arquivo e descobre o que ele contém.

## Ao final deste notebook você terá

1. lido `dados/abate_bovinos.csv` com a biblioteca padrão de Python, sem pandas;
2. contado os registros e convertido a coluna `valor` de texto para número;
3. usado um conjunto para provar que a série está toda na mesma unidade;
4. agrupado o abate por ano e por trimestre com dicionário;
5. lido um traceback de verdade, provocado de propósito;
6. respondido, com o dado na mão, se existe sazonalidade na série.

> **Sem pandas neste notebook.** Pandas entra na Aula 03. Fazer na mão agora é o que
> torna visível o que o pandas vai passar a fazer por você depois.

## 1. Onde está o arquivo

A célula abaixo funciona nos dois lugares onde este notebook roda: no repositório
clonado (onde o CSV está em `../dados/`) e no Google Colab (onde não existe `../dados/`,
então o arquivo é baixado da versão publicada do repositório).

Ela não é enfeite de portabilidade: sem isso, metade da turma abre o notebook no Colab e
trava na primeira célula com `FileNotFoundError`.

In [ ]:
import csv
import os
import urllib.request

ARQUIVO = "abate_bovinos.csv"
CAMINHO_LOCAL = os.path.join("..", "dados", ARQUIVO)
URL_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
             "main/dados/" + ARQUIVO)

if os.path.exists(CAMINHO_LOCAL):
    CAMINHO = CAMINHO_LOCAL
else:
    # Estamos no Colab: baixa uma vez para o disco local da sessao.
    CAMINHO = ARQUIVO
    if not os.path.exists(CAMINHO):
        urllib.request.urlretrieve(URL_BRUTA, CAMINHO)

print("lendo de:", CAMINHO)

## 2. Ler o arquivo

`csv.reader` devolve cada linha como uma **lista de textos**. `next()` consome a primeira
linha, que é o cabeçalho, e deixa o leitor posicionado no primeiro registro de dado.

O `with` fecha o arquivo mesmo que o código falhe no meio do bloco.

In [ ]:
with open(CAMINHO, encoding="utf-8") as arquivo:
    leitor = csv.reader(arquivo)
    cabecalho = next(leitor)
    registros = list(leitor)

print("cabecalho:", cabecalho)
print("registros:", len(registros))
print("primeiro: ", registros[0])
print("ultimo:   ", registros[-1])

São 117 registros, de `1997-T1` a `2026-T1`.

**Repare no período: `2025-T4` é o quarto _trimestre_ de 2025, não um mês.** São quatro
observações por ano, não doze. Guarde esse reparo: o TAPI do parceiro pede projeção
mensal, e o dado aberto não tem mês. A Aula 02 abre exatamente aí.

## 3. Tudo que sai do arquivo é texto

O CSV não guarda número, guarda caracteres. Somar antes de converter é o primeiro erro
clássico do módulo: com texto, o `+` concatena.

In [ ]:
bruto = registros[-1][1]

print("valor bruto:", repr(bruto))
print("tipo:       ", type(bruto).__name__)
print("concatenado:", repr(bruto + bruto))
print("convertido: ", float(bruto))
print("em toneladas:", float(bruto) / 1_000)

### O valor que pode não estar lá

O contrato do arquivo (`dados/README.md`) diz que `valor` vem vazio quando o IBGE marca a
observação como ausente ou suprimida. Nos CSVs versionados hoje não existe nenhuma linha
assim, mas o leitor precisa tratar o caso: os arquivos podem ser regenerados, e
`float("")` levanta `ValueError` no meio do laço, sem dizer qual período era.

**Ausente não é zero.** Trocar o vazio por `0.0` afirma que o Brasil não abateu nenhum
bovino naquele trimestre. A lista de faltantes existe para o dado ausente continuar
visível como ausente.

In [ ]:
valores = []
faltantes = []

for periodo, valor, unidade in registros:
    if not valor:            # string vazia e falsa em Python
        faltantes.append(periodo)
        continue
    valores.append((periodo, float(valor)))

print("convertidos:", len(valores))
print("faltantes:  ", faltantes)

## 4. Conjunto: uma verificação de qualidade em uma linha

Um conjunto não tem ordem nem repetição. Aplicado à coluna `unidade`, ele responde
"quantas unidades diferentes existem neste arquivo?".

Se a resposta fosse 2, somar a coluna `valor` seria somar quilograma com litro.

In [ ]:
unidades = {linha[2] for linha in registros}

print("unidades distintas:", unidades)
print("quantas:           ", len(unidades))
assert len(unidades) == 1, "serie com mais de uma unidade: nao pode ser somada"

## 5. Dicionário: agrupar por ano

Sem dicionário, somar por ano exigiria varrer os 117 registros uma vez por ano, e saber a
lista de anos de antemão. Com dicionário, a chave é o próprio ano encontrado no dado: uma
única passagem resolve, e a lista de anos aparece como efeito colateral.

`por_ano.get(ano, 0.0)` devolve `0.0` na primeira vez que um ano aparece, em vez de
levantar `KeyError`. É o que permite escrever o acumulador em uma linha.

In [ ]:
por_ano = {}

for periodo, valor in valores:
    ano = periodo.split("-")[0]          # '2025-T4' -> '2025'
    por_ano[ano] = por_ano.get(ano, 0.0) + valor

print("anos:", len(por_ano))
print("2025:", por_ano["2025"])
print("1997:", por_ano["1997"])
print("crescimento 1997 -> 2025: %.2f vezes" % (por_ano["2025"] / por_ano["1997"]))

### A armadilha do ano incompleto

O dicionário tem 30 chaves, mas 2026 só tem o primeiro trimestre no arquivo. Comparar
2026 com qualquer ano completo é comparar um trimestre com quatro.

Achar o "ano de maior abate" sem filtrar isso é o tipo de erro que não levanta exceção
nenhuma: ele simplesmente devolve a resposta errada com cara de resposta certa.

In [ ]:
trimestres_por_ano = {}
for periodo, _ in valores:
    ano = periodo.split("-")[0]
    trimestres_por_ano[ano] = trimestres_por_ano.get(ano, 0) + 1

completos = {ano: total for ano, total in por_ano.items()
             if trimestres_por_ano[ano] == 4}

print("anos completos:", len(completos))
print("anos parciais: ", sorted(set(por_ano) - set(completos)))
print("maior abate (so anos completos):", max(completos, key=completos.get))

## 6. Agrupar por trimestre: existe sazonalidade?

Mesma técnica, outra chave. Em vez do ano, o trimestre. Se a média de um trimestre for
sistematicamente maior que a dos outros, a série tem sazonalidade, e isso vira **feature**
na Aula 04.

Repare que T1 tem uma observação a mais que os demais (o `2026-T1`), e essa observação
extra é recente, portanto alta. Mesmo com esse viés a favor, veja onde T1 fica.

In [ ]:
por_trimestre = {}

for periodo, valor in valores:
    trimestre = periodo.split("-")[1]    # '2025-T4' -> 'T4'
    por_trimestre.setdefault(trimestre, []).append(valor)

for trimestre in sorted(por_trimestre):
    serie = por_trimestre[trimestre]
    media = sum(serie) / len(serie)
    print("%s  n=%d  media=%15.0f kg" % (trimestre, len(serie), media))

## 7. Ler o traceback

A próxima célula **falha de propósito**. Não conserte antes de ler a mensagem.

`csv.DictReader` devolve cada linha como dicionário, indexado por chave em vez de posição.
As chaves são as do cabeçalho: `periodo`, `valor` e `unidade`. Vamos pedir `mes`.

In [ ]:
with open(CAMINHO, encoding="utf-8") as arquivo:
    for linha in csv.DictReader(arquivo):
        print(linha["mes"])
        break

### O que essa mensagem está dizendo

```
Traceback (most recent call last):
  File ..., line 3, in <module>
    print(linha["mes"])
          ~~~~~^^^^^^^
KeyError: 'mes'
```

Leia **de baixo para cima**:

- a última linha dá o tipo (`KeyError`) e a causa: a chave pedida, entre aspas;
- as linhas acima dão o caminho até o ponto exato;
- os sinais `~~~~~^^^` apontam a subexpressão culpada: `linha["mes"]`, não o `print`.

`KeyError` não diz que a chave está errada, diz que ela **não está lá**. E não está porque
o arquivo não tem mês: tem trimestre.

Em Javascript, `linha.mes` devolveria `undefined`, o programa seguiria, e o erro
apareceria muito depois, longe da causa. Numa cadeia de três modelos, falhar cedo e perto
da causa é a diferença entre uma hora e um dia de depuração.

**Conserte agora:** troque `"mes"` por uma das três chaves reais e rode de novo.

## 8. Desafio

Responda na célula abaixo, no código, não em comentário.

1. Em quantos por cento o abate do trimestre de maior média supera o de menor média?
2. Existe algum ano completo em que o abate caiu em relação ao ano anterior? Quais?
3. A LDC precisa de previsão para os próximos 24 meses. Quantas observações desta série
   isso significa, e o que você responderia ao parceiro sobre a diferença entre o que ele
   pediu e o que a fonte permite medir?

A pergunta 3 não tem resposta em código: escreva-a como texto na variável `resposta_3`.
Ela é o rascunho do que entra na **ART.1 Entendimento do negócio**, que a Sprint 1 fecha
em 14/08.

In [ ]:
# 1. diferenca percentual entre o maior e o menor trimestre
medias = {t: sum(v) / len(v) for t, v in por_trimestre.items()}
maior, menor = max(medias, key=medias.get), min(medias, key=medias.get)
print("maior: %s  menor: %s  diferenca: %.1f%%"
      % (maior, menor, (medias[maior] / medias[menor] - 1) * 100))

# 2. anos completos em que o abate caiu em relacao ao anterior
anos = sorted(completos)
quedas = [ano for anterior, ano in zip(anos, anos[1:])
          if completos[ano] < completos[anterior]]
print("anos de queda:", quedas)

# 3. escreva a sua resposta aqui, em uma ou duas frases
resposta_3 = ""
print("resposta_3:", repr(resposta_3))